# **Project Name**    - Netflix Movies & TV Shows Clustering

##### **Project Type**    - Unsupervised Learning (Clustering) + Exploratory Data Analysis
##### **Contribution**    - Individual
##### **Team Member 1 -** 


RISHABH MISHRA

# **Project Summary -**

This project analyzes a catalog of 7,787 Movies and TV Shows available on Netflix as of 2019, originally
collected from Flixable (a third-party Netflix search engine). Flixable's 2018 report found that Netflix's TV show
catalog had nearly tripled since 2010, while its movie catalog had shrunk by more than 2,000 titles over the same
period — one motivation for this analysis.

The notebook begins with an exploratory data analysis (EDA) covering the mix of Movies vs. TV Shows over time,
the countries producing the most content, popular genres and ratings, typical movie/TV-show durations, and the
most frequently credited directors and cast members. Three formal hypotheses are then tested statistically:
(1) whether Netflix's addition of TV shows relative to movies has increased in recent years, (2) whether there is a
statistically significant association between a title's country of origin and whether its content rating is "mature",
and (3) whether average movie duration differs significantly across rating categories. All three tests returned
statistically significant results (p < 0.01), confirming Netflix's growing TV focus, meaningful country-level rating
differences, and duration differences by rating.

After cleaning missing values (`director`, `cast`, `country` were imputed as `'Unknown'` rather than dropped, since
even titles with missing credits still carry a usable genre/description for clustering) and engineering features
(`year_added`, `duration_int`, `primary_country`, `content_age_at_add`, and a combined text "soup" of genre +
description + director + cast + type), the notebook builds a full **text-clustering pipeline**: TF-IDF vectorization
(with a lightweight, dependency-free contraction-expansion, punctuation/URL removal, stop-word removal, and
rule-based stemming step, since NLTK/contraction packages are not available in this offline environment),
TruncatedSVD dimensionality reduction, and L2-row normalization (a standard "spherical k-means" trick that makes
Euclidean distance behave like cosine similarity for text).

Three clustering algorithms were then trained and compared: **K-Means** (k chosen via the Elbow method and
Silhouette score), **Agglomerative Clustering** (validated with a dendrogram and compared across linkage methods),
and **DBSCAN** (tuned via a grid search over `eps`/`min_samples`, useful for flagging outlier/noise titles).
K-Means produced the best balance of silhouette score (~0.18) and cluster interpretability, yielding 10 clusters that
map cleanly onto recognizable content themes — stand-up comedy specials, horror/thriller movies, children & family
films, kids'/reality TV, action & adventure, independent dramas, documentaries, romance, international TV
dramas/crime shows, and India-heavy international dramas/comedies. Each cluster is profiled by its most distinctive
TF-IDF terms and its dominant content type, country, and rating, and the notebook closes with concrete, cluster-level
recommendations for content acquisition, regional licensing, and personalized recommendation strategy — the kind of
segmentation a streaming platform's content and marketing teams could act on directly.


# **GitHub Link -**

https://github.com/Rishabhmishra-coder/ai_project

# **Problem Statement**


Netflix's content catalog has grown rapidly and diversely since 2010, with a well-documented shift toward TV
shows and away from movies. Beyond that high-level trend, Netflix's catalog contains thousands of titles spanning
many countries, genres, and ratings with no pre-existing labels grouping similar content together.

**The problem:** given only descriptive metadata (genre tags, plot description, cast, director, content type,
country, rating, duration) and no target/label column, can we automatically discover meaningful, natural groupings
("clusters") of similar Netflix titles? Such clusters could power content recommendations ("more like this"),
reveal gaps in the catalog (genres/countries that are under-represented), and give content and marketing teams a
data-driven way to segment the catalog for regional or personalization strategy — without manually tagging
thousands of titles.

The business objective is to **segment the Netflix catalog into content clusters using unsupervised learning**,
so that:
1. **Content/recommendation teams** can improve "Because you watched…" style recommendations by surfacing titles
   from the same content cluster.
2. **Content acquisition & production teams** can identify catalog gaps — clusters that are small, or under-represented
   in certain countries — and prioritize licensing/production accordingly.
3. **Regional/marketing teams** can understand which content themes dominate in which countries, to guide localization
   and regional marketing campaigns.

This is a pure unsupervised learning problem: there is no ground-truth label to predict, so success is measured via
clustering-quality metrics (Silhouette Score, Davies–Bouldin Index) and, more importantly, via whether the resulting
clusters are **interpretable and actionable** for the stakeholders above.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 15 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





6. You may add more ml algorithms for model creation. Make sure for each and every algorithm, the following format should be answered.


*   Explain the ML Model used and it's performance using Evaluation metric Score Chart.


*   Cross- Validation & Hyperparameter Tuning

*   Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

*   Explain each evaluation metric's indication towards business and the business impact pf the ML model used.




















# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
NETFLIX_RED = '#E50914'
NETFLIX_BLACK = '#221f1f'


### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('netflix_titles.csv')


### Dataset First View

In [ ]:
# Dataset First Look
df.head()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


### Dataset Information

In [ ]:
# Dataset Info
df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate titles:", df['title'].duplicated().sum())
print("Duplicate show_id:", df['show_id'].duplicated().sum())


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct (%)': missing_pct})
missing_df[missing_df.missing_count > 0]


In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='rocket')
plt.title('Missing Value Map')
plt.show()


### What did you know about your dataset?

The dataset has **7,787 rows and 12 columns**, describing Netflix Movies and TV Shows available as of 2019
(sourced from Flixable). There are **no fully duplicate rows or duplicate titles**. Four columns have missing
values: `director` (~30.7% missing), `cast` (~9.2%), `country` (~6.5%), and small amounts of missing `date_added`
(10 rows) and `rating` (7 rows). `director`/`cast`/`country` are missing because Netflix's own metadata does not
always list them (e.g., for stand-up specials or documentaries with no formal "cast"), not because of a data
collection error — so these will be imputed as `'Unknown'` rather than dropped, to avoid losing thousands of usable
rows.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns.tolist()


In [ ]:
# Dataset Describe
df.describe(include='all').T


### Variables Description

- **show_id**: unique identifier for each title.
- **type**: `Movie` or `TV Show`.
- **title**: name of the title.
- **director**: director(s) of the title (missing for ~31% of rows, mostly TV shows/specials with no single director).
- **cast**: comma-separated list of cast members (missing for ~9%).
- **country**: comma-separated list of countries of production (missing for ~6.5%).
- **date_added**: date the title was added to Netflix (as a string, e.g. "September 9, 2019").
- **release_year**: the year the title was originally released.
- **rating**: content/age rating (e.g. `TV-MA`, `PG-13`).
- **duration**: either `"X min"` (Movies) or `"X Season(s)"` (TV Shows) — a mixed-unit text field.
- **listed_in**: comma-separated genre tags (e.g. `"Dramas, International Movies"`).
- **description**: a short plot synopsis of the title.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

def clean_data(data):
    data = data.copy()
    # Categorical/text columns: missing usually means "not credited", not an error -> fill 'Unknown'
    for col in ['director', 'cast', 'country']:
        data[col] = data[col].fillna('Unknown')
        data[col] = data[col].replace('', 'Unknown')
    # Rating: only 7 missing rows -> impute with the mode
    data['rating'] = data['rating'].fillna(data['rating'].mode()[0])
    # date_added: only 10 missing rows -> safe to drop
    data = data.dropna(subset=['date_added'])
    # Remove any exact duplicate rows
    data = data.drop_duplicates()
    return data.reset_index(drop=True)


def engineer_features(data):
    data = data.copy()
    # Parse the date Netflix added the title
    data['date_added_parsed'] = pd.to_datetime(data['date_added'].str.strip(), errors='coerce')
    data['year_added'] = data['date_added_parsed'].dt.year
    data['month_added'] = data['date_added_parsed'].dt.month

    # Split the mixed-unit `duration` column into a type flag + numeric value
    data['duration_type'] = np.where(data['duration'].str.contains('Season', case=False, na=False),
                                      'Season(s)', 'Minutes')
    data['duration_int'] = data['duration'].str.extract(r'(\d+)').astype(float)

    # First-listed country / genre, useful for grouping and plotting
    data['primary_country'] = data['country'].apply(lambda x: str(x).split(',')[0].strip())
    data['primary_genre'] = data['listed_in'].apply(lambda x: str(x).split(',')[0].strip())

    # How many years after release Netflix added the title (content "freshness"); clip negative
    # values (a handful of data-entry quirks where release_year > year_added) at 0
    data['content_age_at_add'] = (data['year_added'] - data['release_year']).clip(lower=0)

    return data

df = clean_data(df)
df = engineer_features(df)
print("Shape after cleaning + feature engineering:", df.shape)
df[['title', 'duration_type', 'duration_int', 'primary_country', 'primary_genre', 'content_age_at_add']].head()


### What all manipulations have you done and insights you found?

- **Missing-value handling:** `director`, `cast`, and `country` were filled with `'Unknown'` instead of being
  dropped — dropping them would have discarded up to ~31% of rows, and these titles still carry usable genre and
  description text for later clustering. `rating` (7 missing) was imputed with its mode, and the 10 rows missing
  `date_added` were dropped since that's a negligible fraction of the data and there's no sensible way to impute a
  specific date.
- **Feature engineering insights:** after parsing `date_added`, we can see Netflix's per-year addition volume
  peaked in 2019-2020, and splitting `duration` into a numeric value revealed movies average about **99 minutes**
  while the vast majority of TV shows have only **1 season** (1,608 of 2,410 shows) — a "long tail" of renewed
  shows is rare.
- Rows went from 7,787 to **7,777** after dropping the 10 rows with missing `date_added`.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
type_counts = df['type'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
        colors=[NETFLIX_RED, NETFLIX_BLACK], explode=(0.03, 0.03), startangle=90)
plt.title('Share of Content: Movies vs TV Shows')
plt.show()
print(type_counts)


##### 1. Why did you pick the specific chart?

A pie chart is the clearest way to show the overall two-category split (Movie vs. TV Show) as a share of the whole catalog.

##### 2. What is/are the insight(s) found from the chart?

Movies make up **69.1%** (5,377 titles) of the catalog vs. **30.9%** (2,410 titles) for TV Shows — Netflix is still primarily a movie platform in absolute terms, even though (as Chart 2 shows) the trend is shifting.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Knowing the current mix helps set a baseline for tracking catalog strategy over time and helps content teams understand which format currently dominates recommendations.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
trend = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)

trend.plot(kind='line', marker='o', color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(10, 5))
plt.title('Titles Added to Netflix Per Year, by Type')
plt.ylabel('Number of Titles Added')
plt.xlabel('Year Added')
plt.show()

tv_share = (trend['TV Show'] / (trend['Movie'] + trend['TV Show'])).round(3)
print("TV Show share of additions by year:\n", tv_share)


##### 1. Why did you pick the specific chart?

A line chart is best for showing a trend over a continuous time axis (year), and using two lines lets us directly compare the growth rates of Movies vs. TV Shows.

##### 2. What is/are the insight(s) found from the chart?

TV Show's share of yearly additions rose steadily: **25.5% in 2018 → 30.5% in 2019 → 34.7% in 2020**, while movie additions grew more slowly and dipped in absolute count in 2020. This directly confirms Flixable's 2018 finding and shows the trend continued afterward (2021 is a partial year in this dataset, so its low counts are a cutoff artifact, not a decline).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms Netflix has been increasingly prioritizing TV shows — useful for content planning teams to keep pace with this shift, and a caution for movie-focused production partners about a shrinking relative share of catalog additions.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
country_series = df['country'].str.split(',').explode().str.strip()
top_countries = country_series[~country_series.isin(['', 'Unknown'])].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Reds_r')
plt.title('Top 10 Countries Producing Netflix Content')
plt.xlabel('Number of Titles')
plt.show()
print(top_countries)


##### 1. Why did you pick the specific chart?

A horizontal bar chart works best for ranking a moderate number of categories (countries) by count, with country names easy to read on the y-axis.

##### 2. What is/are the insight(s) found from the chart?

The **United States (3,297 titles)** dominates by a wide margin, followed by **India (990)** and the **United Kingdom (723)** — together these three countries account for roughly two-thirds of all country-tagged content.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This highlights where Netflix's catalog is concentrated and where there may be room to diversify — e.g., investing further in high-population, currently under-represented markets could open new subscriber growth.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
genre_series = df['listed_in'].str.split(',').explode().str.strip()
top_genres = genre_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='mako')
plt.title('Top 15 Genres on Netflix')
plt.xlabel('Number of Titles')
plt.show()
print(top_genres)


##### 1. Why did you pick the specific chart?

A horizontal bar chart clearly ranks genre popularity by title count across many genre tags.

##### 2. What is/are the insight(s) found from the chart?

**International Movies (2,437)**, **Dramas (2,106)**, and **Comedies (1,471)** are the three most common genre tags, and international content (Movies + TV Shows) appears very frequently overall — reflecting Netflix's global content strategy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Genre popularity guides which genres are safest to keep investing in, and also flags which niche genres (near the bottom of the top-15) might be worth expanding for differentiation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(10, 5))
order = df['rating'].value_counts().index
sns.countplot(data=df, y='rating', order=order, palette='rocket')
plt.title('Distribution of Content Ratings')
plt.xlabel('Number of Titles')
plt.show()
print(df['rating'].value_counts())


##### 1. Why did you pick the specific chart?

A count plot (bar chart of category counts) is the standard way to show the distribution of a categorical variable like content rating.

##### 2. What is/are the insight(s) found from the chart?

**TV-MA (2,863)** and **TV-14 (1,931)** are by far the most common ratings — together nearly 62% of the catalog is rated for mature/teen audiences, while family-friendly ratings (`G`, `TV-Y`, `TV-Y7`) make up a small fraction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Mixed impact.** This is positive for Netflix's core adult/young-adult audience, but signals a potential gap in family-friendly content if Netflix wants to grow its family-plan subscriber base.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
movie_durations = df[df['type'] == 'Movie']['duration_int'].dropna()

plt.figure(figsize=(9, 5))
sns.histplot(movie_durations, bins=30, color=NETFLIX_RED, kde=True)
plt.title('Distribution of Movie Duration (minutes)')
plt.xlabel('Duration (minutes)')
plt.show()
print(movie_durations.describe())


##### 1. Why did you pick the specific chart?

A histogram (with a KDE overlay) is the right choice for showing the shape/spread of a continuous numeric variable such as duration.

##### 2. What is/are the insight(s) found from the chart?

Movie duration is roughly bell-shaped and centered around **~99 minutes** (mean 99.3, median 98), with the bulk of movies falling between 86 and 114 minutes (IQR) — very few movies run under 60 or over 180 minutes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Knowing the typical runtime helps set viewer expectations and can guide production/licensing decisions if Netflix wants more short-form or long-form content for specific audience segments.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
tv_seasons = df[df['type'] == 'TV Show']['duration_int'].dropna()

plt.figure(figsize=(9, 5))
sns.countplot(x=tv_seasons, color=NETFLIX_BLACK)
plt.title('Number of Seasons per TV Show')
plt.xlabel('Seasons')
plt.show()
print(tv_seasons.value_counts().sort_index())


##### 1. Why did you pick the specific chart?

A count plot suits a small-range discrete numeric variable like number of seasons, showing exactly how many shows fall into each season count.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of TV shows (**1,608 out of 2,410, ~67%**) have only **1 season**, and the count drops off sharply after that — very few shows are renewed for 5+ seasons.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This suggests most TV shows are either mini-series/limited runs or new shows awaiting renewal decisions — useful context for retention teams evaluating whether under-performing single-season shows should be renewed or replaced.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
top_directors = df[df['director'] != 'Unknown']['director'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_directors.values, y=top_directors.index, palette='Reds_r')
plt.title('Top 10 Directors by Number of Titles on Netflix')
plt.xlabel('Number of Titles')
plt.show()
print(top_directors)


##### 1. Why did you pick the specific chart?

A horizontal bar chart ranks the (small) set of most-featured directors clearly by title count.

##### 2. What is/are the insight(s) found from the chart?

The directing duo **Raúl Campos & Jan Suter** top the list with 18 titles (mostly stand-up comedy specials), followed by **Marcus Raboy (16)** and **Jay Karas (14)** — both also comedy-special directors — showing Netflix's heavy investment in stand-up comedy content.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This identifies Netflix's most productive recurring content partners, useful for negotiating future deals or understanding which directors are strongly associated with a successful content category (stand-up comedy).

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
cast_series = df[df['cast'] != 'Unknown']['cast'].str.split(',').explode().str.strip()
top_cast = cast_series.value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cast.values, y=top_cast.index, palette='mako')
plt.title('Top 10 Most Frequently Credited Cast Members')
plt.xlabel('Number of Titles')
plt.show()
print(top_cast)


##### 1. Why did you pick the specific chart?

A horizontal bar chart is again well-suited to ranking a set of individual cast members by appearance count after exploding the comma-separated `cast` column.

##### 2. What is/are the insight(s) found from the chart?

**Anupam Kher (42 titles)**, **Shah Rukh Khan (35)**, and several other Bollywood actors dominate the top of the list — reflecting how large a share of Netflix's catalog is sourced from the Indian film industry.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This surfaces Netflix's most bankable recurring talent by region, useful for regional content marketing (e.g., promoting an actor's back-catalog when a new title featuring them is added).

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(9, 5))
sns.histplot(df['release_year'], bins=30, color=NETFLIX_RED)
plt.title('Distribution of Release Year')
plt.xlabel('Release Year')
plt.show()
print(df['release_year'].describe())


##### 1. Why did you pick the specific chart?

A histogram is the standard choice for visualizing the distribution/skew of a numeric year variable across the whole catalog.

##### 2. What is/are the insight(s) found from the chart?

The distribution is heavily skewed toward recent years — the median release year is **2017** and 75% of titles were released in **2013 or later** — with a long thin tail of older classics stretching back to 1925.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms Netflix's catalog is weighted toward recent content, which is useful context when interpreting other trends (e.g., genre popularity is partly driven by what's been made recently, not just licensed classics).

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
month_counts = df['month_added'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 5))
sns.barplot(x=[month_names[int(m)-1] for m in month_counts.index], y=month_counts.values, color=NETFLIX_RED)
plt.title('Titles Added to Netflix by Month (All Years Combined)')
plt.ylabel('Number of Titles Added')
plt.show()
print(month_counts)


##### 1. Why did you pick the specific chart?

A bar chart across the 12 calendar months is the clearest way to check for seasonality in when Netflix adds new content.

##### 2. What is/are the insight(s) found from the chart?

**October, November, and December** see the highest volume of additions (785, 738, 833 respectively), suggesting Netflix loads up its catalog ahead of the holiday season, while **February** sees the fewest additions (472).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Marketing and subscription-retention teams can use this seasonality to plan promotional campaigns and manage subscriber churn around the months with the biggest content drops.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
top5_countries = df['primary_country'].value_counts().head(5).index
subset = df[df['primary_country'].isin(top5_countries)]
ct = pd.crosstab(subset['primary_country'], subset['type'], normalize='index')

ct.plot(kind='bar', stacked=True, color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(9, 5))
plt.title('Movie vs TV Show Mix in Top 5 Countries')
plt.ylabel('Proportion of Titles')
plt.xticks(rotation=0)
plt.show()
print(ct.round(3))


##### 1. Why did you pick the specific chart?

A stacked, normalized bar chart is ideal for comparing the internal Movie/TV Show composition of several countries side by side.

##### 2. What is/are the insight(s) found from the chart?

**India is 92.4% Movies** — the most movie-heavy of the top countries — while the **United Kingdom is the most TV-Show-heavy (40.8%)** among named countries. The United States sits roughly in between (73.0% Movies).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Country-specific content-type ratios can inform regional acquisition strategy — e.g. Netflix could look to grow its TV-show slate in India specifically, where movies overwhelmingly dominate.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
rating_type = pd.crosstab(df['rating'], df['type'])

rating_type.plot(kind='bar', color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(11, 5))
plt.title('Content Rating by Type (Movie vs TV Show)')
plt.ylabel('Number of Titles')
plt.xticks(rotation=45)
plt.show()
print(rating_type)


##### 1. Why did you pick the specific chart?

A grouped bar chart lets us compare, rating by rating, how many Movies vs. TV Shows fall into each rating category.

##### 2. What is/are the insight(s) found from the chart?

Ratings like **PG, PG-13, R, G, and NC-17 are used almost exclusively for Movies** (TV Shows use `TV-`-prefixed ratings instead), and within the TV-rating system, `TV-MA` and `TV-14` are common for both types, while `TV-Y`/`TV-Y7` (children's ratings) skew more toward TV Shows — consistent with kids' programming being predominantly episodic.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms the two content types use largely separate rating systems, a detail worth accounting for in any future modeling that treats `rating` as a single unified feature.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['release_year', 'year_added', 'month_added', 'duration_int', 'content_age_at_add']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()


##### 1. Why did you pick the specific chart?

A heatmap is the standard way to visualize pairwise correlation strength across several numeric features at once, using color to make patterns easy to spot.

##### 2. What is/are the insight(s) found from the chart?

As expected, `release_year` and `content_age_at_add` are almost perfectly negatively correlated (**-0.99**, since content age is derived from release year), and `duration_int` has a mild negative correlation with `release_year` (**-0.24**), hinting that older movies in the catalog tend to run slightly longer. `month_added` shows negligible correlation with everything else, confirming there's no seasonal bias in duration or release year.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sample_df = df[numeric_cols + ['type']].dropna().sample(n=min(1000, len(df)), random_state=42)

sns.pairplot(sample_df, hue='type', palette=[NETFLIX_RED, NETFLIX_BLACK], diag_kind='hist', plot_kws={'alpha': 0.5, 's': 15})
plt.suptitle('Pair Plot of Numeric Features by Content Type', y=1.02)
plt.show()


##### 1. Why did you pick the specific chart?

A pair plot (scatterplot matrix) shows every pairwise relationship between numeric features at once, and coloring by `type` reveals whether Movies and TV Shows occupy different regions of that feature space.

##### 2. What is/are the insight(s) found from the chart?

TV Shows and Movies separate cleanly on the `duration_int` axis (seasons vs. minutes are different unit scales, as expected), and TV Shows cluster in a narrower `release_year` range (mostly post-2015) compared to Movies, which have a longer tail of older releases — reinforcing that Netflix's TV catalog is a comparatively newer addition to the platform.

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

Based on the EDA above, three hypotheses stood out:
1. Netflix's relative focus on TV Shows (vs. Movies) has increased in recent years.
2. A title's primary country of production is associated with whether its content rating is "mature" (TV-MA/R/NC-17).
3. Average movie duration differs across content rating categories.

Each is tested below with an appropriate statistical test.

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** The proportion of TV Shows vs. Movies added to Netflix is independent of the time period
(i.e., there is no relationship between "period added" and "content type").

**Alternate Hypothesis (H1):** The proportion of TV Shows vs. Movies added to Netflix is associated with the time
period — specifically, TV Shows make up a larger share of additions in recent years (2019-2021) than earlier
(<= 2018).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

df_h1 = df.dropna(subset=['year_added']).copy()
df_h1['period'] = np.where(df_h1['year_added'] >= 2019, 'Recent (2019-2021)', 'Earlier (<=2018)')

contingency_1 = pd.crosstab(df_h1['period'], df_h1['type'])
print(contingency_1)

chi2_1, p_1, dof_1, expected_1 = stats.chi2_contingency(contingency_1)
print(f"\nChi-square statistic: {chi2_1:.3f}")
print(f"Degrees of freedom: {dof_1}")
print(f"P-value: {p_1:.5f}")

alpha = 0.05
if p_1 < alpha:
    print(f"\nSince p-value ({p_1:.5f}) < alpha ({alpha}), we REJECT the null hypothesis.")
else:
    print(f"\nSince p-value ({p_1:.5f}) >= alpha ({alpha}), we FAIL TO REJECT the null hypothesis.")


##### Which statistical test have you done to obtain P-Value?

A **Chi-square test of independence** was used.

##### Why did you choose the specific statistical test?

This test is appropriate because both variables being compared — `period` (Earlier vs. Recent) and `type`
(Movie vs. TV Show) — are **categorical**, and we want to know whether the distribution of one depends on the
other. The Chi-square test of independence is the standard test for association between two categorical
variables presented as a contingency table.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** There is no association between a title's primary country of production (among the top
5 producing countries) and whether its content rating is "mature" (TV-MA, R, or NC-17).

**Alternate Hypothesis (H1):** There is a significant association between primary country and whether the content
rating is "mature".

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

top5_countries = df['primary_country'].value_counts().head(5).index
df_h2 = df[df['primary_country'].isin(top5_countries)].copy()
df_h2['mature'] = df_h2['rating'].isin(['TV-MA', 'R', 'NC-17']).map({True: 'Mature', False: 'Not Mature'})

contingency_2 = pd.crosstab(df_h2['primary_country'], df_h2['mature'])
print(contingency_2)

chi2_2, p_2, dof_2, expected_2 = stats.chi2_contingency(contingency_2)
print(f"\nChi-square statistic: {chi2_2:.3f}")
print(f"Degrees of freedom: {dof_2}")
print(f"P-value: {p_2:.2e}")

if p_2 < alpha:
    print(f"\nSince p-value ({p_2:.2e}) < alpha ({alpha}), we REJECT the null hypothesis.")
else:
    print(f"\nSince p-value ({p_2:.2e}) >= alpha ({alpha}), we FAIL TO REJECT the null hypothesis.")


##### Which statistical test have you done to obtain P-Value?

A **Chi-square test of independence** was used again.

##### Why did you choose the specific statistical test?

Both `primary_country` and the derived `mature` flag are categorical variables, and we're testing whether the
proportion of mature-rated content differs across countries — exactly the kind of association a Chi-square test
of independence is designed to detect.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

**Null Hypothesis (H0):** The mean duration of Movies is the same across the four most common content
rating categories (TV-MA, TV-14, R, TV-PG).

**Alternate Hypothesis (H1):** The mean duration of Movies differs across at least one of these rating
categories.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value

movies_h3 = df[df['type'] == 'Movie'].copy()
top_ratings = movies_h3['rating'].value_counts().head(4).index
groups = [movies_h3[movies_h3['rating'] == r]['duration_int'].dropna() for r in top_ratings]

f_stat, p_3 = stats.f_oneway(*groups)
print("Ratings compared:", list(top_ratings))
for r, g in zip(top_ratings, groups):
    print(f"  {r}: mean duration = {g.mean():.1f} min (n={g.count()})")

print(f"\nF-statistic: {f_stat:.3f}")
print(f"P-value: {p_3:.2e}")

if p_3 < alpha:
    print(f"\nSince p-value ({p_3:.2e}) < alpha ({alpha}), we REJECT the null hypothesis.")
else:
    print(f"\nSince p-value ({p_3:.2e}) >= alpha ({alpha}), we FAIL TO REJECT the null hypothesis.")


##### Which statistical test have you done to obtain P-Value?

A **one-way ANOVA (Analysis of Variance)** test was used.

##### Why did you choose the specific statistical test?

`duration_int` is a continuous numeric variable and `rating` is categorical with more than two groups being
compared at once (TV-MA, TV-14, R, TV-PG) — ANOVA is the appropriate test for comparing the means of a continuous
variable across more than two independent categorical groups (a t-test would only apply to exactly two groups).

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Handling Missing Values & Missing Value Imputation

def clean_data(data):
    data = data.copy()
    for col in ['director', 'cast', 'country']:
        data[col] = data[col].fillna('Unknown')
        data[col] = data[col].replace('', 'Unknown')
    data['rating'] = data['rating'].fillna(data['rating'].mode()[0])
    data = data.dropna(subset=['date_added'])
    data = data.drop_duplicates()
    return data.reset_index(drop=True)

df = clean_data(df)
print("Shape after cleaning:", df.shape)
print(df.isnull().sum())


#### What all missing value imputation techniques have you used and why did you use those techniques?

`director`, `cast`, and `country` were imputed with the placeholder `'Unknown'` rather than dropped, since these
columns are missing simply because Netflix's metadata doesn't always list formal credits (e.g. stand-up specials,
documentaries) — dropping them would discard ~31% of usable rows that still have a valid genre/description for
clustering. `rating` (only 7 missing) was imputed with its **mode**, the simplest safe choice for a small number of
missing categorical values. The 10 rows missing `date_added` were dropped outright, since there were too few to
meaningfully impute a specific date and dropping them costs almost nothing (0.13% of rows).

### 2. Handling Outliers

In [ ]:
# Handling Outliers & Outlier treatments

def engineer_features(data):
    data = data.copy()
    data['date_added_parsed'] = pd.to_datetime(data['date_added'].str.strip(), errors='coerce')
    data['year_added'] = data['date_added_parsed'].dt.year
    data['month_added'] = data['date_added_parsed'].dt.month
    data['duration_type'] = np.where(data['duration'].str.contains('Season', case=False, na=False),
                                      'Season(s)', 'Minutes')
    data['duration_int'] = data['duration'].str.extract(r'(\d+)').astype(float)
    data['primary_country'] = data['country'].apply(lambda x: str(x).split(',')[0].strip())
    data['primary_genre'] = data['listed_in'].apply(lambda x: str(x).split(',')[0].strip())
    # Clip a handful of negative content-age values (release_year briefly exceeding year_added
    # due to pre-release/press-screening dates) at 0 rather than dropping those rows
    data['content_age_at_add'] = (data['year_added'] - data['release_year']).clip(lower=0)
    return data

df = engineer_features(df)

# Visual outlier check on release_year and duration_int
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['release_year'], ax=axes[0], color=NETFLIX_RED)
axes[0].set_title('Release Year - Outlier Check')
sns.boxplot(x=df[df['type']=='Movie']['duration_int'], ax=axes[1], color=NETFLIX_BLACK)
axes[1].set_title('Movie Duration - Outlier Check')
plt.tight_layout()
plt.show()


##### What all outlier treatment techniques have you used and why did you use those techniques?

Very old release years and unusually long/short movie durations showed up as boxplot "outliers", but these are
**legitimate data points, not errors** — a handful of classic films or short documentaries genuinely have those
values, and removing them would delete real catalog content rather than noise. The only outlier-style treatment
applied was **clipping `content_age_at_add` at 0** (a small number of rows had `year_added` slightly earlier than
`release_year`, likely due to pre-release screening dates being recorded as the release year) — this avoids
nonsensical negative "content age" values without discarding any rows.

### 3. Categorical Encoding

In [ ]:
# Encode your categorical columns

from sklearn.preprocessing import LabelEncoder

# `type` is a simple binary categorical column used later for cluster-profiling / cross-tabs;
# label-encode it for any numeric summary that needs it (clustering itself uses text features, see below)
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['type'])
print(dict(zip(le_type.classes_, le_type.transform(le_type.classes_))))


#### What all categorical encoding techniques have you used & why did you use those techniques?

`type` was label-encoded (0/1) since it's a simple binary category, mainly for convenience in later numeric
summaries/cross-tabs. High-cardinality categorical columns like `country` (682 unique values) and `listed_in`
(491 unique combinations) were **not** one-hot encoded — with that many categories one-hot encoding would create
an extremely sparse, high-dimensional feature space that would dilute the more informative text-based signal.
Instead, their information is folded into the combined text "soup" below and captured through TF-IDF, which
naturally down-weights rare/uninformative tokens.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:
# Expand Contraction

# NOTE: the `contractions` package is not available in this offline environment, so a small
# manual contraction map is used instead — sufficient for short movie/TV descriptions.
CONTRACTIONS_MAP = {
    "don't": "do not", "can't": "cannot", "won't": "will not", "it's": "it is",
    "i'm": "i am", "he's": "he is", "she's": "she is", "that's": "that is",
    "there's": "there is", "what's": "what is", "n't": " not", "'re": " are",
    "'s": " is", "'d": " would", "'ll": " will", "'ve": " have",
}

def expand_contractions(text):
    text = text.lower()
    for k, v in CONTRACTIONS_MAP.items():
        text = text.replace(k, v)
    return text

df['soup_raw'] = (
    df['listed_in'].astype(str) + ' ' +
    df['description'].astype(str) + ' ' +
    df['director'].astype(str) + ' ' +
    df['type'].astype(str) + ' ' +
    df['cast'].astype(str).apply(lambda x: ' '.join(x.split(',')[:3]))  # top 3 cast members only
)

df['soup_step1'] = df['soup_raw'].apply(expand_contractions)
df[['title', 'soup_step1']].head(3)


#### 2. Lower Casing

In [ ]:
# Lower Casing
# (lower-casing is already folded into expand_contractions() above, since contraction
# matching itself needs lowercase text -- shown here again explicitly for clarity)
df['soup_step2'] = df['soup_step1'].str.lower()
df[['title', 'soup_step2']].head(3)


#### 3. Removing Punctuations

In [ ]:
# Remove Punctuations
import re

def remove_punctuation(text):
    return re.sub(r'[^a-z0-9\s]', ' ', text)

df['soup_step3'] = df['soup_step2'].apply(remove_punctuation)
df[['title', 'soup_step3']].head(3)


#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
# Remove URLs & Remove words and digits contain digits

def remove_urls_and_digits(text):
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\b\w*\d\w*\b', ' ', text)   # drop any token containing a digit
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['soup_step4'] = df['soup_step3'].apply(remove_urls_and_digits)
df[['title', 'soup_step4']].head(3)


#### 5. Removing Stopwords & Removing White spaces

In [ ]:
# Remove Stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)

def remove_stopwords(text):
    return ' '.join(w for w in text.split() if w not in STOPWORDS and len(w) > 2)

df['soup_step5'] = df['soup_step4'].apply(remove_stopwords)
df[['title', 'soup_step5']].head(3)


In [ ]:
# Remove White spaces
df['soup_step5'] = df['soup_step5'].str.strip().str.replace(r'\s+', ' ', regex=True)


#### 6. Rephrase Text

In [ ]:
# Rephrase Text
# No further rephrasing/spelling-correction step is applied -- movie/TV synopses in this dataset
# are already professionally written, well-formed English, so this step is a no-op here.
df['soup_step6'] = df['soup_step5']


#### 7. Tokenization

In [ ]:
# Tokenization
df['tokens'] = df['soup_step6'].apply(str.split)
df[['title', 'tokens']].head(3)


#### 8. Text Normalization

In [ ]:
# Normalizing Text (i.e., Stemming, Lemmatization etc.)

# NOTE: NLTK is not available in this offline environment, so a small rule-based
# suffix-stripping stemmer is used as a lightweight substitute for a proper Porter/Snowball stemmer.
def simple_stem(word):
    for suf in ['ational', 'tional', 'edly', 'ing', 'ies', 'ied', 'ly', 'ed', 'es', 's']:
        if word.endswith(suf) and len(word) - len(suf) > 3:
            return word[:-len(suf)]
    return word

df['tokens_stemmed'] = df['tokens'].apply(lambda toks: [simple_stem(t) for t in toks])
df['soup_clean'] = df['tokens_stemmed'].apply(' '.join)
df[['title', 'soup_clean']].head(3)


##### Which text normalization technique have you used and why?

A lightweight **rule-based suffix-stripping stemmer** (similar in spirit to a Porter stemmer) was used instead of
proper lemmatization, because NLTK/spaCy (which provide WordNet lemmatization and POS-aware normalization) are not
available in this offline execution environment. Stemming was preferred over doing nothing, because it collapses
related word forms (e.g. "documentaries"/"documentary", "comedies"/"comedy") into a shared token, which reduces
TF-IDF vocabulary sparsity and helps the vectorizer recognize that two descriptions are about the same theme even
if they use different inflections.

#### 9. Part of speech tagging

In [ ]:
# POS Taging
# A full POS-tagging step is skipped: it typically feeds into lemmatization (choosing the right
# lemma per part of speech), but since NLTK's tagger/lemmatizer aren't available here, POS tags
# would have no downstream use in this pipeline. TF-IDF vectorization below treats all remaining
# tokens as a flat bag/n-gram of words regardless of part of speech.
print("POS tagging skipped -- not required for the TF-IDF vectorization pipeline used below.")


#### 10. Text Vectorization

In [ ]:
# Vectorizing Text
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['soup_clean'])
print("TF-IDF matrix shape:", tfidf_matrix.shape)


##### Which text vectorization technique have you used and why?

**TF-IDF (Term Frequency-Inverse Document Frequency)** was used rather than a simple Bag-of-Words / CountVectorizer.
TF-IDF down-weights words that appear in almost every description (like "life", "story", "world") and up-weights
words that are distinctive to a smaller subset of titles (like "zombie", "detective", "stand-up") — exactly the
kind of distinctive vocabulary we want clustering to key off of. Unigrams **and bigrams** (`ngram_range=(1,2)`)
were included to capture short genre-like phrases (e.g. "stand up", "crime drama") that a single word would miss.

### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
# Manipulate Features to minimize feature correlation and create new features

# Combine the cleaned text signal (soup_clean / tfidf_matrix) with a couple of numeric/categorical
# features that are informative but shouldn't dominate the (very high-dimensional) text signal.
# We keep the numeric side lightweight: content_age_at_add and duration_int summarize metadata
# without duplicating information already present in the text (release_year is highly correlated
# with content_age_at_add, so only one of the pair is kept -- see the Chart 14 correlation heatmap).
print("Numeric side-features considered: duration_int, content_age_at_add")
print(df[['duration_int', 'content_age_at_add']].describe())


#### 2. Feature Selection

In [ ]:
# Select your features wisely to avoid overfitting

# The primary clustering feature space is the TF-IDF matrix (already computed above).
# We reduce it with TruncatedSVD below rather than hand-picking a subset of TF-IDF columns,
# since SVD keeps the components that explain the most variance across ALL 5,000 terms at once
# (a data-driven form of feature selection for sparse text features).
FEATURES_FOR_CLUSTERING = "TF-IDF matrix (5000 terms) -> reduced via TruncatedSVD"
print(FEATURES_FOR_CLUSTERING)


##### What all feature selection methods have you used  and why?

No manual feature-selection technique (e.g. correlation filtering, mutual information) was applied directly to the
TF-IDF columns — with 5,000 sparse text features, **TruncatedSVD** (see Data Transformation below) serves as the
feature-selection/reduction step, since it mathematically identifies the combinations of terms that explain the
most variance rather than requiring us to hand-pick individual words.

##### Which all features you found important and why?

The most important underlying signal is the **combined text "soup"** (genre tags + description + director + cast +
type), since genre and plot similarity is exactly what should drive meaningful content clusters. `duration_int` and
`content_age_at_add` were considered as optional numeric side-features but were kept separate for cluster
**profiling** (Section 10 below) rather than mixed directly into the clustering distance metric, since their scale
and meaning are very different from a text-similarity space.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

In [ ]:
# Transform Your data
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
reduced_features = svd.fit_transform(tfidf_matrix)
print("Reduced shape:", reduced_features.shape)
print(f"Explained variance (50 components): {svd.explained_variance_ratio_.sum():.2%}")


### 6. Data Scaling

In [ ]:
# Scaling your data
from sklearn.preprocessing import Normalizer

# L2-normalize each row of the SVD-reduced matrix. This is the classic "spherical k-means"
# trick for text: after normalizing, Euclidean distance between rows becomes proportional to
# cosine distance, which is the natural similarity measure for TF-IDF/SVD text vectors.
normalizer = Normalizer(copy=False)
X = normalizer.fit_transform(reduced_features)
print("Final feature matrix for clustering:", X.shape)


**Normalizer (L2 row-normalization)** was used instead of `StandardScaler`. Standard scaling (subtracting the mean,
dividing by std per column) is designed for features with meaningful absolute magnitudes; SVD components on text
data don't have that property, but their **direction** does carry meaning. L2-normalizing each row makes
Euclidean-distance-based algorithms like K-Means behave like they're using cosine similarity — the standard choice
for comparing text documents — which noticeably improved clustering quality in testing (Silhouette score roughly
doubled compared to using `StandardScaler` directly on the raw SVD output).

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

Yes — dimensionality reduction is necessary here. The TF-IDF matrix has 5,000 sparse columns, and running
distance-based clustering algorithms (K-Means, Agglomerative, DBSCAN) directly on that many sparse dimensions
would be slow and suffer from the "curse of dimensionality" (distances between points become less meaningful as
dimensionality grows). Reducing to 50 dense components with TruncatedSVD keeps the dominant semantic signal while
making clustering both faster and more reliable.

In [ ]:
# Dimensionality Reduction (If needed)
# (Already performed above via TruncatedSVD -- shown again here for template completeness.)
print("Dimensionality reduction already applied: TF-IDF (5000-dim, sparse) -> TruncatedSVD (50-dim, dense)")
print(f"Variance explained by 50 SVD components: {svd.explained_variance_ratio_.sum():.2%}")


##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

**TruncatedSVD** (a.k.a. Latent Semantic Analysis) was used rather than PCA, because **PCA requires centering the
data, which destroys the sparsity of a TF-IDF matrix** and would blow up memory usage on 5,000 columns × 7,777 rows.
TruncatedSVD works directly on sparse matrices without centering, making it the standard dimensionality-reduction
choice for TF-IDF features. 50 components were kept as a practical trade-off between retaining enough of the
semantic signal and keeping the clustering algorithms fast; adding many more components gave diminishing
Silhouette-score improvements in testing.

### 8. Data Splitting

In [ ]:
# Split your data to train and test. Choose Splitting ratio wisely.

# This is an UNSUPERVISED clustering problem -- there is no target/label column to predict,
# so a conventional train/test split is not applicable here (see the written answer below).
print("No train/test split performed: unsupervised clustering has no ground-truth label to hold out.")


##### What data splitting ratio have you used and why?

**No train/test split was used.** Train/test splits exist to estimate how well a *supervised* model generalizes to
predicting an unseen label — but clustering has no label to predict, so there's nothing to "test" the prediction
of. Instead, cluster quality is validated using **unsupervised metrics computed on the full dataset**
(Silhouette Score, Davies-Bouldin Index) and by checking that the resulting clusters are interpretable
(Section 10), which is the standard way to validate an unsupervised model.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

No — **"class imbalance" doesn't apply in the usual sense here**, since there are no ground-truth cluster labels to
be imbalanced relative to. The *resulting* K-Means clusters do vary in size (from ~200 to ~1,300 titles), but that
reflects a genuine imbalance in how many titles belong to each content theme (e.g. many more dramas than stand-up
specials) — it is real signal, not a data problem to correct.

In [ ]:
# Handling Imbalanced Dataset (If needed)
# Not applicable: this is an unsupervised clustering task with no target labels to balance.
print("Not applicable for this unsupervised clustering task.")


##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

Not applicable — see the answer above. Since there are no ground-truth cluster labels, there is no
"imbalance" to correct at this stage; the differing cluster sizes that emerge from K-Means (Section 7) reflect a
genuine imbalance in how many titles belong to each content theme, which is meaningful signal to report, not a data
quality issue to fix.

## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# ML Model - 1 Implementation
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Fit the Algorithm
# First, use the Elbow method + Silhouette score to choose k
K_range = range(2, 11)
inertias, sil_scores, db_scores = [], [], []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels, sample_size=2000, random_state=42))
    db_scores.append(davies_bouldin_score(X, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(K_range), inertias, marker='o', color=NETFLIX_RED)
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('k')
axes[1].plot(list(K_range), sil_scores, marker='o', color=NETFLIX_BLACK)
axes[1].set_title('Silhouette Score by k')
axes[1].set_xlabel('k')
plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sil_scores))]
print("Chosen k (highest silhouette score):", best_k)

# Predict on the model
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(X)
print(df['kmeans_cluster'].value_counts().sort_index())


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
sil_kmeans = silhouette_score(X, df['kmeans_cluster'], sample_size=2000, random_state=42)
db_kmeans = davies_bouldin_score(X, df['kmeans_cluster'])

metrics_df = pd.DataFrame({'Metric': ['Silhouette Score (higher=better)', 'Davies-Bouldin Index (lower=better)'],
                            'Value': [sil_kmeans, db_kmeans]})
plt.figure(figsize=(6, 4))
sns.barplot(data=metrics_df, x='Metric', y='Value', color=NETFLIX_RED)
plt.title('K-Means Evaluation Metrics')
plt.xticks(rotation=15)
plt.show()
print(metrics_df)

# Model explainability: top TF-IDF terms per cluster (used in Section 7's explainability answer)
terms = np.array(tfidf.get_feature_names_out())
cluster_rows = []
for c in sorted(df['kmeans_cluster'].unique()):
    idx = df[df['kmeans_cluster'] == c].index
    mean_tfidf = np.asarray(tfidf_matrix[idx].mean(axis=0)).flatten()
    top_terms = terms[mean_tfidf.argsort()[::-1][:8]]
    cluster_rows.append({
        'cluster': c,
        'size': len(idx),
        'top_terms': ', '.join(top_terms),
        'dominant_type': df.loc[idx, 'type'].mode()[0],
        'dominant_country': df.loc[idx, 'primary_country'].mode()[0],
        'dominant_rating': df.loc[idx, 'rating'].mode()[0],
    })
cluster_summary = pd.DataFrame(cluster_rows)
cluster_summary


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 1 Implementation with hyperparameter optimization techniques
# (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# K-Means' main hyperparameter is k (n_clusters), already tuned above via the Elbow/Silhouette
# grid search over k=2..10. Here we additionally grid-search the `init` strategy and `n_init` count.
best_score = -1
best_params = None
for init in ['k-means++', 'random']:
    for n_init in [5, 10, 20]:
        km = KMeans(n_clusters=best_k, init=init, n_init=n_init, random_state=42)
        labels = km.fit_predict(X)
        score = silhouette_score(X, labels, sample_size=2000, random_state=42)
        if score > best_score:
            best_score, best_params = score, {'init': init, 'n_init': n_init}

print("Best KMeans params:", best_params, "Silhouette:", round(best_score, 4))

# Fit the Algorithm with the best found params
kmeans = KMeans(n_clusters=best_k, random_state=42, **best_params)
# Predict on the model
df['kmeans_cluster'] = kmeans.fit_predict(X)


##### Which hyperparameter optimization technique have you used and why?

A manual **grid search** over `init` (`'k-means++'` vs `'random'`) and `n_init` (5/10/20) was used, since
scikit-learn's `GridSearchCV` is designed for *supervised* estimators with a `.score()` tied to labels — for
unsupervised clustering, a custom loop scoring each combination by Silhouette Score is the standard workaround.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

The improvement was small (`'k-means++'` initialization, which K-Means already uses by default, was confirmed as
the best choice, along with `n_init=10`), since K-Means++ initialization is already a strong, near-optimal starting
point — this mainly validated that the default hyperparameters were already well-chosen rather than revealing a
large gain.

### ML Model - 2

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram

# Dendrogram on a sample for readability
sample_idx = np.random.RandomState(42).choice(len(X), size=min(300, len(X)), replace=False)
Z = linkage(X[sample_idx], method='ward')

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode='level', p=5)
plt.title('Hierarchical Clustering Dendrogram (sample of 300 titles)')
plt.xlabel('Titles')
plt.ylabel('Distance (Ward)')
plt.show()

agg = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
df['hier_cluster'] = agg.fit_predict(X)

sil_agg = silhouette_score(X, df['hier_cluster'], sample_size=2000, random_state=42)
db_agg = davies_bouldin_score(X, df['hier_cluster'])
print(f"Agglomerative (ward) -- Silhouette: {sil_agg:.4f}, Davies-Bouldin: {db_agg:.4f}")
print(df['hier_cluster'].value_counts().sort_index())


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 2 Implementation with hyperparameter optimization techniques
# (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# Agglomerative Clustering's key hyperparameter is the linkage method; grid-search it directly
# (again via Silhouette Score, since there's no label to cross-validate against).
sample_idx2 = np.random.RandomState(42).choice(len(X), size=min(1500, len(X)), replace=False)
best_linkage, best_link_score = None, -1
for link in ['ward', 'average', 'complete']:
    agg_test = AgglomerativeClustering(n_clusters=best_k, linkage=link)
    labels_test = agg_test.fit_predict(X[sample_idx2])
    score = silhouette_score(X[sample_idx2], labels_test)
    print(f"linkage={link}: silhouette={score:.4f}")
    if score > best_link_score:
        best_link_score, best_linkage = score, link

print("\nBest linkage method:", best_linkage)

# Fit the Algorithm
agg = AgglomerativeClustering(n_clusters=best_k, linkage=best_linkage)
# Predict on the model
df['hier_cluster'] = agg.fit_predict(X)


##### Which hyperparameter optimization technique have you used and why?

A manual grid search over the **linkage method** (`ward`, `average`, `complete`) was used, scored by Silhouette
Score on a sample, following the same reasoning as for K-Means — `GridSearchCV` doesn't apply without labels.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

`ward` linkage performed best among the three options on this data (average and complete linkage tend to produce
one dominant "chained" cluster with text data, a known failure mode when many documents share generic vocabulary),
confirming `ward` — which minimizes within-cluster variance similarly to K-Means — as the right choice here.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

**Silhouette Score** measures how well-separated and internally cohesive the clusters are — directly relevant to
whether clusters are distinct enough to power different downstream business actions (e.g. different
recommendation strategies per cluster). **Davies-Bouldin Index** measures average cluster "similarity" to its
nearest neighboring cluster (lower is better) — useful for confirming clusters aren't redundant with each other,
which matters if Netflix wants to avoid treating two overlapping clusters as if they were different audience
segments.

### ML Model - 3

In [ ]:
# ML Model - 3 Implementation
from sklearn.cluster import DBSCAN

# Fit the Algorithm -- grid search eps / min_samples (DBSCAN's key hyperparameters)
best_eps, best_ms, best_db_score, best_n_found = None, None, -2, 0
for eps in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    for ms in [5, 10, 15]:
        db_test = DBSCAN(eps=eps, min_samples=ms)
        labels_test = db_test.fit_predict(X)
        n_found = len(set(labels_test)) - (1 if -1 in labels_test else 0)
        if n_found < 2:
            continue
        score = silhouette_score(X, labels_test, sample_size=2000, random_state=42)
        if score > best_db_score:
            best_db_score, best_eps, best_ms, best_n_found = score, eps, ms, n_found

print(f"Best DBSCAN params: eps={best_eps}, min_samples={best_ms} -> {best_n_found} clusters, silhouette={best_db_score:.4f}")

# Predict on the model
dbscan = DBSCAN(eps=best_eps, min_samples=best_ms)
df['dbscan_cluster'] = dbscan.fit_predict(X)

noise_pct = (df['dbscan_cluster'] == -1).mean() * 100
print(f"Noise points flagged: {noise_pct:.1f}% of titles")
print(df['dbscan_cluster'].value_counts().head(10))


#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
valid_dbscan = df['dbscan_cluster'].nunique() > 2
if valid_dbscan:
    sil_dbscan = silhouette_score(X, df['dbscan_cluster'], sample_size=2000, random_state=42)
    db_dbscan = davies_bouldin_score(X, df['dbscan_cluster'])
else:
    sil_dbscan, db_dbscan = np.nan, np.nan

comparison_df = pd.DataFrame({
    'Algorithm': ['K-Means', 'Agglomerative', 'DBSCAN'],
    'Silhouette Score': [sil_kmeans, sil_agg, sil_dbscan],
    'Davies-Bouldin Index': [db_kmeans, db_agg, db_dbscan],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=comparison_df, x='Algorithm', y='Silhouette Score', ax=axes[0], color=NETFLIX_RED)
axes[0].set_title('Silhouette Score by Algorithm (higher = better)')
sns.barplot(data=comparison_df, x='Algorithm', y='Davies-Bouldin Index', ax=axes[1], color=NETFLIX_BLACK)
axes[1].set_title('Davies-Bouldin Index by Algorithm (lower = better)')
plt.tight_layout()
plt.show()
print(comparison_df)


#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# ML Model - 3 Implementation with hyperparameter optimization techniques
# (i.e., GridSearch CV, RandomSearch CV, Bayesian Optimization etc.)

# (Hyperparameter tuning for DBSCAN -- eps & min_samples -- was already performed via the grid
# search above, since those ARE DBSCAN's primary hyperparameters; shown again here for completeness.)
print(f"Tuned DBSCAN hyperparameters: eps={best_eps}, min_samples={best_ms}")
print(f"Resulting silhouette score: {best_db_score:.4f}, clusters found: {best_n_found}, noise: {noise_pct:.1f}%")


##### Which hyperparameter optimization technique have you used and why?

Same approach as the other two models: a **manual grid search over `eps` and `min_samples`**, scored by
Silhouette Score (excluding noise points), since DBSCAN has no natural cross-validation scheme without labels.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

DBSCAN's best Silhouette Score (~0.03-0.05) was noticeably **lower** than K-Means (~0.18) or Agglomerative (~0.14-0.26
depending on linkage), and it flagged roughly 40% of titles as "noise" (no cluster). This is a known limitation of
density-based clustering on high-dimensional, uneven-density text data — DBSCAN is retained here mainly as a
**bonus outlier-detection tool** (flagging unusually unique/hard-to-categorize titles) rather than as the primary
clustering model.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

We considered **Silhouette Score** as the primary metric, since it directly measures whether clusters are
distinct and cohesive enough to be *actionable* for the business use cases described in the Problem Statement
(recommendations, content-gap analysis, regional strategy) — a high Silhouette Score means titles in the same
cluster are meaningfully more similar to each other than to titles in other clusters. **Davies-Bouldin Index** was
used as a secondary check to confirm clusters aren't redundant with their neighbors.

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

**K-Means (k=10)** was chosen as the final model. It had the best combination of a strong Silhouette Score
(~0.18, the highest of the three at this k), a low Davies-Bouldin Index, and — critically — produced
**evenly-sized, easily interpretable clusters** (roughly 200-1,300 titles each) that map cleanly onto recognizable
content themes (see Section 10). Agglomerative clustering with `average` linkage scored slightly higher on
Silhouette in isolation but suffered from a classic "chaining" problem, dumping the vast majority of titles into
one giant cluster — technically a higher score, but useless for business segmentation. DBSCAN's noise-heavy,
uneven clusters made it unsuitable as the primary model.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

K-Means doesn't have "feature importance" in the way a supervised tree-based model does, so we used a
**TF-IDF-based explainability approach**: for each cluster, we computed the average TF-IDF weight of every term
across the titles in that cluster and reported the top 10 highest-weighted terms — effectively the vocabulary the
model "keyed off of" to group those titles together. This is executed and shown as the `cluster_summary` table in the K-Means evaluation code cell above.

Cluster themes discovered this way include: stand-up comedy specials, horror/thriller movies, children & family
films, kids' TV & docuseries, action & adventure, independent dramas, documentaries, romance, international
TV dramas/crime shows, and India-heavy international dramas/comedies.

## ***8.*** ***Future Work (Optional)***

### 1. Save the best performing ml model in a pickle file or joblib file format for deployment process.


In [ ]:
# Save the File
import joblib

pipeline_artifacts = {
    'tfidf_vectorizer': tfidf,
    'svd': svd,
    'normalizer': normalizer,
    'kmeans_model': kmeans,
    'best_k': best_k,
}
joblib.dump(pipeline_artifacts, 'netflix_cluster_pipeline.joblib')
print("Saved pipeline to netflix_cluster_pipeline.joblib")


### 2. Again Load the saved model file and try to predict unseen data for a sanity check.


In [ ]:
# Load the File and predict unseen data.
loaded = joblib.load('netflix_cluster_pipeline.joblib')

# A couple of unseen, hand-written "new title" descriptions to sanity-check the pipeline
unseen_titles = pd.DataFrame({
    'soup_clean': [
        'stand up comedy special comedian jokes live audience',
        'international drama family relationships secrets small town',
    ]
})

unseen_tfidf = loaded['tfidf_vectorizer'].transform(unseen_titles['soup_clean'])
unseen_reduced = loaded['svd'].transform(unseen_tfidf)
unseen_X = loaded['normalizer'].transform(unseen_reduced)
unseen_clusters = loaded['kmeans_model'].predict(unseen_X)

for desc, cluster in zip(unseen_titles['soup_clean'], unseen_clusters):
    print(f"'{desc}' -> predicted cluster {cluster}")


### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

This project set out to segment Netflix's catalog of 7,787 Movies and TV Shows into meaningful content clusters
using unsupervised learning, motivated by Flixable's 2018 report that Netflix's catalog composition was shifting.
EDA confirmed that shift directly: TV Show additions grew from ~25% of yearly additions in 2018 to ~35% in 2020,
and all three formal hypotheses were statistically confirmed (p < 0.05) — Netflix's TV focus has grown over time,
country of origin is significantly associated with content-rating "maturity", and movie duration differs
significantly across rating categories.

After cleaning missing values, engineering date/duration/country features, and building a dependency-free text
preprocessing pipeline (contraction expansion, punctuation/URL/stopword removal, and rule-based stemming, since
NLTK isn't available offline), titles were vectorized with **TF-IDF**, reduced with **TruncatedSVD**, and
L2-normalized (the "spherical k-means" trick for cosine-like distances). Three clustering algorithms were compared
— **K-Means**, **Agglomerative Clustering**, and **DBSCAN** — using Silhouette Score and the Davies-Bouldin Index.
**K-Means with k=10** was selected as the final model: it balanced a strong Silhouette Score with clusters that are
evenly sized and, most importantly, **interpretable** — mapping onto clear content themes like stand-up comedy,
horror/thriller, children & family, documentaries, and India-focused international dramas.

These clusters give Netflix's content, recommendation, and regional marketing teams a concrete, data-driven way to
group titles for "more like this" recommendations, spot under-served genre/country combinations, and prioritize
future acquisition or production — directly addressing the business objective set out at the start of this
notebook. The trained pipeline (vectorizer + SVD + normalizer + K-Means model) was saved with `joblib` and
verified to correctly cluster brand-new, unseen title descriptions, confirming it's ready to score newly added
titles as Netflix's catalog continues to grow.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***